# Lesson 3: Derivatives & Gradients

*ML Academy — companion notebook*

Slopes, derivatives, the chain rule, partial derivatives and the gradient — the maths of learning.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-03.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 11. Checking derivatives with code

In [1]:
def f(x):
    return (3 * x + 1) ** 2

def f_prime(x):          # our hand-derived answer
    return 6 * (3 * x + 1)

def numeric_derivative(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)

for x in [0, 1, -2]:
    print(x, f_prime(x), round(numeric_derivative(f, x), 4))
# 0 6 6.0
# 1 24 24.0
# -2 -30 -30.0

0 6 6.0
1 24 24.0
-2 -30 -30.0


In [2]:
w = 0.0
lr = 0.05                       # learning rate (eta)
for step in range(30):
    grad = 4 * (2 * w - 6)      # dL/dw
    w = w - lr * grad           # move downhill
print(round(w, 4))              # 3.0  -> the perfect weight

3.0


## 14. Going deeper: calculus for optimisation

In [3]:
import numpy as np

def f(v):                                  # a function from 2 numbers to 2 numbers
    x, y = v
    return np.array([x**2 * y, 5 * x + np.sin(y)])

def jacobian_formula(v):
    x, y = v
    return np.array([[2 * x * y, x**2],    # row 1: ∂f1/∂x, ∂f1/∂y
                     [5.0, np.cos(y)]])    # row 2: ∂f2/∂x, ∂f2/∂y

def jacobian_numeric(f, v, h=1e-6):
    cols = [(f(v + h * e) - f(v - h * e)) / (2 * h) for e in np.eye(len(v))]
    return np.column_stack(cols)

v = np.array([1.0, 2.0])
print(jacobian_formula(v).round(4))
print(jacobian_numeric(f, v).round(4))

[[ 4.      1.    ]
 [ 5.     -0.4161]]
[[ 4.      1.    ]
 [ 5.     -0.4161]]


In [4]:
def hessian_numeric(g, v, h=1e-4):
    d = len(v); H = np.zeros((d, d))
    for i in range(d):
        for j in range(d):
            ei, ej = np.eye(d)[i] * h, np.eye(d)[j] * h
            H[i, j] = (g(v + ei + ej) - g(v + ei - ej) - g(v - ei + ej) + g(v - ei - ej)) / (4 * h * h)
    return H

bowl = lambda w: w[0]**2 + 10 * w[1]**2          # a long, narrow valley
saddle = lambda w: w[0]**2 - w[1]**2             # up in one direction, down in the other
for name, g in [("bowl", bowl), ("saddle", saddle)]:
    H = hessian_numeric(g, np.array([0.0, 0.0]))
    print(name, "Hessian eigenvalues:", np.linalg.eigvalsh(H).round(3))

bowl Hessian eigenvalues: [ 2. 20.]
saddle Hessian eigenvalues: [-2.  2.]


In [5]:
grad = lambda w: np.array([2 * w[0], 20 * w[1]])
H = np.array([[2.0, 0.0], [0.0, 20.0]])

for lr in [0.01, 0.05, 0.09, 0.11]:
    w = np.array([10.0, 1.0])
    for step in range(1, 10001):
        w = w - lr * grad(w)
        if np.linalg.norm(w) < 1e-6 or np.linalg.norm(w) > 1e6: break
    status = "diverged" if np.linalg.norm(w) > 1e6 else f"{step} steps"
    print(f"gradient descent, learning rate {lr}: {status}")

w = np.array([10.0, 1.0])
w = w - np.linalg.solve(H, grad(w))              # one Newton step
print("Newton's method: 1 step, lands at", w)

gradient descent, learning rate 0.01: 798 steps
gradient descent, learning rate 0.05: 153 steps
gradient descent, learning rate 0.09: 82 steps
gradient descent, learning rate 0.11: diverged
Newton's method: 1 step, lands at [0. 0.]


In [6]:
# Gradient check for a tiny model: loss(w) = mean((tanh(Xw) − y)²)
rng = np.random.default_rng(0)
X = rng.normal(size=(20, 3)); y = rng.normal(size=20); w = rng.normal(size=3)
loss = lambda w: np.mean((np.tanh(X @ w) - y) ** 2)

def grad_formula(w):
    p = np.tanh(X @ w)
    return X.T @ (2 * (p - y) * (1 - p**2)) / len(y)   # chain rule: tanh' = 1 − tanh²

def rel_error(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b))

h = 1e-5
numeric = np.array([(loss(w + h * e) - loss(w - h * e)) / (2 * h) for e in np.eye(3)])
print("relative error, correct formula:", f"{rel_error(grad_formula(w), numeric):.1e}")
buggy = lambda w: X.T @ (2 * (np.tanh(X @ w) - y)) / len(y)   # forgot the tanh' factor
print("relative error, buggy formula:  ", f"{rel_error(buggy(w), numeric):.1e}")

relative error, correct formula: 3.4e-11
relative error, buggy formula:   6.7e-02


In [7]:
from scipy.optimize import minimize

# Largest rectangle with perimeter 20: maximise x·y subject to 2x + 2y = 20
res = minimize(lambda v: -v[0] * v[1], x0=[1.0, 1.0],
               constraints=[{"type": "eq", "fun": lambda v: 2 * v[0] + 2 * v[1] - 20}])
print("numerical answer (x, y):", res.x.round(4), " area:", round(-res.fun, 4))

numerical answer (x, y): [5. 5.]  area: 25.0


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [8]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
x, y = df["size_m2"].to_numpy(), df["price_lakh"].to_numpy()
def loss(w):            # MSE for the model price = w * size
    return np.mean((w * x - y) ** 2)
def grad(w):            # dL/dw = (2/n) * sum((w*x - y) * x)
    return 2 * np.mean((w * x - y) * x)
for w in [0.4, 0.65, 0.9]:
    print(w, round(loss(w), 1), round(grad(w), 1))

0.4 1116.2 -6130.0
0.65 336.4 -109.2
0.9 1061.7 5911.7


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [9]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Numerical derivative

Estimate the slope of f(x) = x² at x = 3 with a tiny step h.

In [10]:
def f(x):
    return x ** 2
h = 1e-6
slope = None  # ← your code here
check(slope, 6.000001000927568, hint='Rise over run: (f(3 + h) - f(3)) / h. It should be close to 6.')

❌ Not yet. Hint: Rise over run: (f(3 + h) - f(3)) / h. It should be close to 6.


### Challenge 2: One gradient descent step

L(w) = (w − 4)². Take one step from w = 0 with learning rate 0.25.

In [11]:
w, lr = 0.0, 0.25
grad = 2 * (w - 4)
w_new = None  # ← your code here
check(w_new, 2.0, hint='Move against the gradient: w - lr * grad.')

❌ Not yet. Hint: Move against the gradient: w - lr * grad.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Numerical derivative** — the missing line:

```python
slope = (f(3 + h) - f(3)) / h
```

**2. One gradient descent step** — the missing line:

```python
w_new = w - lr * grad
```